In [308]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

%matplotlib inline

In [309]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

#Veriyi bilgisayarımdaki path'den bulup okudu

In [310]:
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [311]:
df.shape


(7043, 21)

In [312]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [313]:
pd.to_numeric(df['TotalCharges'], errors='coerce').isnull().sum()


np.int64(11)

In [314]:
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")

#Burda errors="coerce" kullanmak dönüşütürmede sorun yaşarsa NaN yapmasını sağlıyor.

In [315]:
df['Churn'].value_counts(normalize=True)  # yüzde olarak
df['Churn'].value_counts()

#Burdan baktığımda unbalanced bir veriye sahipim gibi duruyor.Bunun kararını modeli oluşturup metrikleri inceledikten sonra karar vericem.

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [316]:
for col in df.select_dtypes(include='str').columns:
    print(df[col].unique())


#Burası sayesinde encode edeceğim kolonların içeriklerine for döngüsü ve unique fonkisyonu ile hem hatasız hem de hızlıca oluşıcam


<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str
<StringArray>
['Female', 'Male']
Length: 2, dtype: str
<StringArray>
['Yes', 'No']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
<StringArray>
['No', 'Yes', 'No internet

#Değişkenlerimi birleştirelim


In [317]:

cols_to_fix =[ 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in cols_to_fix:

          df[col]= df[col].replace("No internet service","No")
df['MultipleLines']= df['MultipleLines'].replace("No phone service","No")


In [318]:
for col in df.select_dtypes(include='str').columns:
    print(df[col].unique())



<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str
<StringArray>
['Female', 'Male']
Length: 2, dtype: str
<StringArray>
['Yes', 'No']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['Yes', 'No']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['Month-to-month', 'One year', 'Two ye

Encode edelim

In [319]:
liste=[ 'MultipleLines','OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies','Partner','Dependents','PhoneService','PaperlessBilling','Churn']

for col in liste:
    df[col] = df[col].replace({"No": 0, "Yes": 1})

Kontrol edelim


In [320]:
liste=[ 'MultipleLines','OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies','Partner','Dependents','PhoneService','PaperlessBilling','Churn']


for i in liste:
    print(df[i].unique())

df['gender'] = df['gender'].replace({"Female": 0, "Male": 1})


[0 1]
[0 1]
[1 0]
[0 1]
[0 1]
[0 1]
[0 1]
[1 0]
[0 1]
[0 1]
[1 0]
[0 1]


Aynı encode işlemlerini yapmaya devam edelim.


In [321]:
print(df['gender'].unique())
#Ne olur ne olmaz elle konrtol

[0 1]


Bu sayede bütün binary kolonlar encode edilmiş oldu.


In [322]:
for col in df.select_dtypes(include='str').columns:
    print(df[col].unique())



<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
<StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str
<StringArray>
[         'Electronic check',              'Mailed check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str


In [323]:
cols_to_encode_nominal = ['InternetService', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=cols_to_encode_nominal)

Sıra Nominal verileri encode etmekte


In [324]:

liste = ['InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No',
         'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year',
         'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)',
         'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
for col in liste:
    df[col] = df[col].replace({False: 0, True: 1})


#Aslında model burdaki true ve false ları zaten 0 ve 1 olarak görecek ama ben yine de yapmak istedim

In [325]:
cols_to_convert = df.select_dtypes(include=['bool', 'object']).columns.tolist()
cols_to_convert.remove('customerID')
df = df.drop('customerID', axis=1)

for col in cols_to_convert:
    df[col] = df[col].astype(int)

#Kendimce sayıya dönüştürmüş olabiliirm ama hala pythona bu sayı demem gerekli bu nedenle encode ettiğim tüm kolonlara veri tipi değişimi uyguladım.
#Ayno zamanda modeli eğitirken kullanmayacağımız customerID kolonunu da attım df den.
#

/var/folders/bh/c7386y_56js2n2360hrm_rq80000gn/T/ipykernel_1892/4094693345.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_to_convert = df.select_dtypes(include=['bool', 'object']).columns.tolist()


In [326]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   Partner                                  7043 non-null   int64  
 3   Dependents                               7043 non-null   int64  
 4   tenure                                   7043 non-null   int64  
 5   PhoneService                             7043 non-null   int64  
 6   MultipleLines                            7043 non-null   int64  
 7   OnlineSecurity                           7043 non-null   int64  
 8   OnlineBackup                             7043 non-null   int64  
 9   DeviceProtection                         7043 non-null   int64  
 10  TechSupport                              7043 non-null   in

In [327]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(df['TotalCharges'].isnull().sum())
#TotalCharges içinde yer alan NaN gözlemleri de 0 a atadım ki 11 tanecik de olsa gözlem kaybetmeyelim.
#Kontrolü de NaN değerlere 0 atadığım için bunların toplammı 0 olmalıdır idyerek kontrol ettim.


0


MODELİ OLUŞTURALIM


In [328]:
X = df.drop('Churn', axis=1)#x değişkenlerini belirleyelim bunlar churn hariç büyün veri kolonları olacak,y değişkeni target yani churn kolonu olacak.
y = df['Churn']
print(X.shape)
print(y.shape)

(7043, 26)
(7043,)


In [329]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, stratify=y, random_state=61)